# UC-DIM-2 — Administrative Hierarchy Navigation

**Persona:** FAO Regional Analyst

**Goal:** Navigate a 2-level administrative dimension (continent → country), retrieve
children of a continent node, inspect ancestor chains, and handle edge cases such as
leaf nodes (no further children) and unknown parent nodes.

**Use case:** UC-DIM-2 — spatial scope selection for regional crop assessments

**Prerequisites:**
- `extension_dimensions` installed and registered
- `world-admin` dimension available (5 continents → 49 countries)

In [ ]:
import os

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
TOKEN = os.environ.get("DYNASTORE_TOKEN", "")
DIM_ID = "world-admin"   # leveled tree: 5 continents -> 49 countries
DIM_BASE = "/dimensions"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30.0, follow_redirects=True)
print(f"BASE_URL: {BASE_URL}")
print(f"DIM_ID  : {DIM_ID}")

## Step 1 — Inspect the admin dimension

Fetch the dimensions listing and confirm `world-admin` is a hierarchical dimension.

In [ ]:
resp = client.get(DIM_BASE)
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

collections = resp.json().get("collections", [])
admin_dim = next((d for d in collections if d["id"] == DIM_ID), None)
assert admin_dim is not None, f"Dimension '{DIM_ID}' not found"

provider = admin_dim.get("provider", {})
print(f"id          : {admin_dim.get('id')}")
print(f"title       : {admin_dim.get('title')}")
print(f"provider    : {provider.get('type')}")
print(f"hierarchical: {provider.get('hierarchical')}")
print(f"description : {admin_dim.get('description', '—')[:80]}")

assert provider.get("hierarchical") is True, "Expected hierarchical=True for world-admin"
print("\nHierarchical dimension confirmed.")

## Step 2 — Navigate Africa children

Request direct children of `AFR` (Africa). Expect country-level nodes (ISO 3166-1 alpha-3 codes).

In [ ]:
resp = client.get(f"{DIM_BASE}/{DIM_ID}/children", params={"parent": "AFR"})
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
afr_children = data.get("features", [])

print(f"Africa child nodes ({len(afr_children)}):")
for node in afr_children[:10]:
    props = node["properties"]
    print(f"  id={node['id']}  label={props.get('title', '—')}")

assert len(afr_children) > 0, "Africa (AFR) should have at least one child country node"
country_ids = [n["id"] for n in afr_children]
print(f"\nCountry IDs: {country_ids[:5]} ...")

## Step 3 — Verify leaf nodes

Countries are leaf nodes in this dimension (no further subdivisions). Requesting children of a country must return an empty list.

In [ ]:
# Pick first country from previous result
leaf_country = country_ids[0]
resp = client.get(f"{DIM_BASE}/{DIM_ID}/children", params={"parent": leaf_country})
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
leaf_children = data.get("features", [])
print(f"{leaf_country} children: {len(leaf_children)}")

assert len(leaf_children) == 0, (
    f"Expected 0 children for leaf country {leaf_country}, got {len(leaf_children)}"
)
print(f"\nLeaf node confirmed: {leaf_country} has no further subdivisions.")

## Step 4 — Ancestor chain (breadcrumb)

Given a country code (e.g. `ETH`), retrieve its ancestors to build a breadcrumb. Expect `AFR` in the chain.

In [ ]:
target = "ETH"  # Ethiopia
resp = client.get(f"{DIM_BASE}/{DIM_ID}/ancestors", params={"member": target})
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
ancestors = data.get("features", [])
ancestor_ids = [n["id"] for n in ancestors]
print(f"Ancestor chain for {target}: {ancestor_ids}")

assert "AFR" in ancestor_ids, (
    f"Expected AFR (Africa) in ancestor chain of {target}; got {ancestor_ids}"
)
print(f"\nBreadcrumb confirmed: {' > '.join(ancestor_ids)} > {target}")

## Edge cases

### Unknown parent — returns empty list

Requesting children of an unknown node must return 200 with an empty list (not 404).

In [ ]:
resp = client.get(f"{DIM_BASE}/{DIM_ID}/children", params={"parent": "DOES_NOT_EXIST"})
print(resp.status_code, resp.text[:200])
# The API returns 200 + empty FeatureCollection for unknown nodes
assert resp.status_code == 200, (
    f"Expected 200 (empty list) for unknown parent, got {resp.status_code}: {resp.text}"
)
data = resp.json()
unknown_children = data.get("features", [])
assert len(unknown_children) == 0, f"Expected empty list, got {len(unknown_children)} items"
print("Empty list (200) confirmed for unknown parent — correct behaviour.")

In [ ]:
# Recursive vs leveled distinction
# A leveled dimension carries dimension:level in each member.
resp = client.get(f"{DIM_BASE}/{DIM_ID}/children", params={"parent": "AFR"})
assert resp.status_code == 200

data = resp.json()
sample_nodes = data.get("features", [])

if sample_nodes:
    first = sample_nodes[0]
    props = first["properties"]
    has_level = "dimension:level" in props
    print(f"Sample node '{first['id']}' properties: {list(props.keys())}")
    print(f"Has dimension:level field: {has_level}")
    if has_level:
        print(f"Level value: {props['dimension:level']}")